# Урок 1.3 — Марковские процессы принятия решений (MDP)
## Учебный проект «RoomBot»: от пустой комнаты до оптимальной политики

Это **полностью готовый к запуску** ноутбук к уроку 1.3. Он собран из всех блоков
кода урока — но здесь они склеены в одну работающую программу, дополнены
визуализациями и подробными комментариями.

**Что мы построим (снизу вверх):**

| § | Что делаем | Результат |
|---|---|---|
| 1 | Мир RoomBot: состояния, действия, награда | модель среды в 15 строк |
| 2 | Таблица переходов со скольжением | стохастическая динамика |
| 3 | Три вида политик: случайная, детерминированная, softmax | правило выбора действия |
| 4 | Эпизоды (rollout) и возврат `G_t` | траектория на сетке |
| 5 | Оценка `V^pi` — Монте-Карло и аналитически | **тепловая карта ценности** |
| 6 | Переход к `Q^pi(s, a)` одним шагом вперёд | 4 карты по действиям |
| 7 | Value Iteration | **картина обучения по итерациям + `V*`** |
| 8 | Влияние `gamma` | серия карт |
| 9 | То же на PyTorch | сверка двух реализаций |
| 10 | Проверка политики симуляцией | средний возврат |

**Мир RoomBot.** Комната 3×3. Бот стоит в клетке `(строка, столбец)` и делает шаг
в одну из четырёх сторон. Пол скользкий: с вероятностью 0.9 бот идёт куда хотел,
с вероятностью 0.1 — в случайную из трёх других сторон. Стена возвращает бота на
место. В клетке `(2, 2)` стоит зарядная станция (`+1`), в клетке `(1, 1)` — лужа
(`-1`); обе завершают эпизод. Каждый шаг стоит `-0.02`, чтобы бот не гулял
бесконечно.

**Задача урока:** найти `V*(s)` — сколько «стоит» каждая клетка, если действовать
оптимально, — и извлечь из неё саму оптимальную политику.

## Установка окружения

Ноутбуку нужны только `numpy` и `matplotlib`; `torch` понадобится один раз в § 9
(этот раздел можно пропустить — остальное от него не зависит).

**Вариант А — уже есть conda-среда курса.** Активируйте её и запускайте:

```bash
conda activate mlagents
jupyter notebook
```

**Вариант Б — отдельная лёгкая среда под этот урок:**

```bash
conda create -n rl-mdp python=3.10 -y
conda activate rl-mdp
pip install numpy matplotlib jupyter
pip install torch --index-url https://download.pytorch.org/whl/cpu   # опционально, для § 9
jupyter notebook
```

**Вариант В — VS Code.** Откройте файл, нажмите «Select Kernel» справа вверху и
выберите интерпретатор нужной среды. Дальше — «Run All».

Ноутбук считается за пару секунд, GPU не нужен: вся «нейросеть» здесь — таблица
из девяти чисел.

In [ ]:
# Единственная ячейка с импортами на весь ноутбук.
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10

print("numpy      ", np.__version__)
print("matplotlib ", plt.matplotlib.__version__)

## § 1. Мир RoomBot: состояние, действие, награда

Вводим ровно три сущности — и сразу в коде.

### 1.1 Состояние S

Состояние — это всё, что боту нужно знать прямо сейчас, чтобы принять решение.
Для RoomBot достаточно двух чисел: строка и столбец. Никакого класса заводить не
нужно — `tuple` уже неизменяем и хешируем, поэтому спокойно кладётся в словари и
множества (а это ровно то, что понадобится для таблиц `V` и `Q`).

In [ ]:
# Состояние RoomBot — это просто пара координат (строка, столбец).
GRID = 3
STATES = [(r, c) for r in range(GRID) for c in range(GRID)]

print(len(STATES))   # 9
print(STATES[0])     # (0, 0)

### 1.2 Действие A

Действий четыре: вверх, вниз, влево, вправо. Держим их как индексы `0..3` — так
их удобно использовать и как ключ словаря, и как индекс массива. Рядом кладём
сдвиг по сетке и человекочитаемую подпись.

In [ ]:
# Действие — индекс 0..3. Рядом храним сдвиг по сетке и подпись для вывода.
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
ACTIONS = (UP, DOWN, LEFT, RIGHT)
MOVES = [(-1, 0), (+1, 0), (0, -1), (0, +1)]   # (дельта строки, дельта столбца)
NAMES = ["вверх", "вниз", "влево", "вправо"]

print(ACTIONS)                       # (0, 1, 2, 3)
print(NAMES[RIGHT], MOVES[RIGHT])    # вправо (0, 1)

### 1.3 Награда R

Награда зависит только от того, в какой клетке бот оказался **после** шага:
станция даёт `+1`, лужа `-1`, любая другая клетка — маленький штраф `-0.02`.

Зачем этот штраф? Без него «топтаться на месте» бесплатно, и агент, у которого
впереди вечность, не торопится к цели. `-0.02` за шаг делает время дорогим:
короткий путь к станции выгоднее длинного.

In [ ]:
BATTERY = (2, 2)   # зарядная станция: +1
PUDDLE  = (1, 1)   # лужа: -1

def reward(s):
    """Награда за то, что бот оказался в клетке s."""
    return {BATTERY: +1.0, PUDDLE: -1.0}.get(s, -0.02)

# Sanity-check:
print(reward(BATTERY))   #  1.0
print(reward(PUDDLE))    # -1.0
print(reward((0, 2)))    # -0.02

## § 2. Переходы: единственная таблица динамики на весь урок

`next_states(s, a)` возвращает распределение следующих состояний — список пар
`(вероятность, состояние)`. К этой функции мы будем возвращаться в § 4, § 5, § 6
и § 7, каждый раз считая по ней ожидания.

Никакой магии: clamp на границы (стена возвращает бота на место) плюс
распределение «0.9 на задуманное направление, оставшиеся 0.1 поровну на три
других».

**Марковское свойство.** Обратите внимание: `next_states` смотрит только на
текущую пару `(s, a)` и ничего не знает об истории. Это и есть марковость —
будущее зависит от прошлого только через настоящее. Именно она позволяет хранить
ценность как таблицу из девяти чисел, а не как функцию от всей траектории.

In [ ]:
SLIP = 0.1   # вероятность проскальзывания

def move(s, a):
    """Клетка назначения; стена возвращает бота на место (clamp)."""
    r, c = s
    dr, dc = MOVES[a]
    return (min(max(r + dr, 0), GRID - 1), min(max(c + dc, 0), GRID - 1))

def next_states(s, a):
    """[(вероятность, следующее состояние), ...] — по одной паре на действие:
    0.9 на задуманное, оставшиеся 0.1 поровну на три остальных."""
    return [((1 - SLIP) if b == a else SLIP / 3, move(s, b)) for b in ACTIONS]

# Sanity-check: из (0,0) вверх упираемся в стену и остаёмся в (0,0) с p=0.9
print(next_states((0, 0), UP))
# Из (1,1) вправо: 0.9 в (1,2), по 0.033... на каждое из трёх других
print(next_states((1, 1), RIGHT))

In [ ]:
# Проверим, что распределение честное: сумма вероятностей = 1 для любой пары (s, a).
worst = max(abs(sum(p for p, _ in next_states(s, a)) - 1.0)
            for s in STATES for a in ACTIONS)
print(f"максимальное отклонение суммы вероятностей от 1: {worst:.1e}")
assert worst < 1e-12, "таблица переходов не нормирована"

## § 3. Политика: как агент выбирает действие

Политика — это правило выбора действия. Посмотрим на три её вида, от самого
примитивного к самому общему.

* **v1, случайная:** каждое действие с вероятностью 1/4. Ничему не учится, но это
  наш нулевой уровень — с ним будем сравнивать всё остальное.
* **v2, детерминированная:** словарь «клетка в действие». В § 7 такую же таблицу
  выдаст Value Iteration, но уже не вручную, а автоматически.
* **v3, стохастическая (softmax):** распределение по действиям, полученное из
  оценок их привлекательности. Температура `tau` управляет жёсткостью: `tau` к
  нулю превращает softmax почти в argmax, большая `tau` — почти в равномерное.

In [ ]:
def policy_random(s, rng):
    """v1 — равномерная политика: каждое из четырёх действий равновероятно."""
    return rng.integers(4)


# v2 — детерминированная политика «вправо пока можешь, потом вниз»:
# словарь «клетка -> действие». В § 7 такую же таблицу выдаст Value Iteration.
fixed_policy = {(r, c): (RIGHT if c < 2 else DOWN) for r, c in STATES}

print(NAMES[fixed_policy[(0, 0)]])   # вправо
print(NAMES[fixed_policy[(0, 2)]])   # вниз


def softmax(q, tau=1.0):
    """v3 — распределение над действиями из вектора Q(s, ·) длины 4.
    tau — «температура»: tau к нулю почти argmax, большая tau почти равномерно."""
    e = np.exp((q - q.max()) / tau)   # вычитаем max ради устойчивости
    return e / e.sum()

# Пока Q-значений нет, но механизм уже работает:
demo_q = np.array([0.2, -0.1, 0.0, 0.9])
print("tau=1.0 :", softmax(demo_q, tau=1.0).round(3))
print("tau=0.1 :", softmax(demo_q, tau=0.1).round(3))   # почти argmax
print("tau=5.0 :", softmax(demo_q, tau=5.0).round(3))   # почти равномерно

## § 4. Эпизод и возврат G

`rollout` прогоняет один эпизод: на каждом шаге спрашиваем политику, разыгрываем
переход по `next_states` и записываем тройку `(s, a, r)`. Эпизод заканчивается
в терминальной клетке (станция или лужа) либо по лимиту шагов.

Возврат — это сумма наград с дисконтом:
`G_t = r_{t+1} + gamma * r_{t+2} + gamma^2 * r_{t+3} + ...`
Коэффициент `gamma` из полуинтервала `[0, 1)` говорит, насколько агента волнует
далёкое будущее.

In [ ]:
TERMINAL = {BATTERY, PUDDLE}   # из этих клеток эпизод уже не продолжается

def rollout(s, policy, rng, horizon=50):
    """Один эпизод: список троек (s, a, r) до терминала или до horizon шагов."""
    traj = []
    for _ in range(horizon):
        a = policy(s, rng)
        probs, outs = zip(*next_states(s, a))       # из § 2
        s_next = outs[rng.choice(4, p=probs)]
        traj.append((s, a, reward(s_next)))         # reward — из § 1.3
        if s_next in TERMINAL:
            break
        s = s_next
    return traj


def discounted_return(traj, gamma=0.95):
    """G_0 — возврат с самого начала эпизода."""
    G = 0.0
    for _s, _a, r in reversed(traj):   # с конца: один проход вместо степеней gamma
        G = r + gamma * G
    return G


rng = np.random.default_rng(7)
traj = rollout((0, 0), policy_random, rng)

for t, (s, a, r) in enumerate(traj):
    print(f"шаг {t}: из {s} идём {NAMES[a]:<7} -> награда {r:+.2f}")
print(f"\nдлина эпизода: {len(traj)} шагов, G_0 = {discounted_return(traj):+.3f}")

### Как это выглядит на сетке

Нарисуем посещённые клетки. Видно и «полезные» шаги, и петли — следствие
скользкого пола: иногда бот идёт не туда, куда собирался.

In [ ]:
def plot_trajectory(traj, title="Эпизод случайного бота", ax=None):
    """Путь бота поверх схемы комнаты."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))

    ax.imshow(np.zeros((GRID, GRID)), cmap="Greys", vmin=0, vmax=1)
    ax.text(BATTERY[1], BATTERY[0], "станция\n+1", ha="center", va="center",
            color="tab:green", fontweight="bold")
    ax.text(PUDDLE[1], PUDDLE[0], "лужа\n-1", ha="center", va="center",
            color="tab:red", fontweight="bold")

    rows = [s[0] for s, _a, _r in traj]
    cols = [s[1] for s, _a, _r in traj]
    ax.plot(cols, rows, "-o", color="tab:cyan", lw=2, ms=9, alpha=0.85, zorder=3)
    ax.plot(cols[0], rows[0], "o", color="white", ms=13, zorder=4)

    ax.set_title(title)
    ax.set_xticks(range(GRID)); ax.set_yticks(range(GRID))
    ax.set_xlabel("столбец"); ax.set_ylabel("строка")
    return ax


plot_trajectory(traj)
plt.show()

## § 5. Функция ценности V: сколько «стоит» клетка

`V^pi(s)` — ожидаемый возврат, если стартовать из `s` и дальше действовать по
политике `pi`. Это одно число на клетку: карта «где хорошо, где плохо».

Посчитаем её двумя способами и убедимся, что они совпадают.

### 5.1 Способ первый: прямой Монте-Карло

Самый честный и самый прямолинейный способ: прогнать тысячи эпизодов и усреднить
получившиеся `G_t` по каждому посещённому состоянию. Ключевой трюк — идти по
траектории **с конца**, накапливая `G = r + gamma * G`: за один проход получаем
`G_t` сразу для всех шагов эпизода.

In [ ]:
def estimate_V(policy, n_episodes=5000, gamma=0.95, seed=0):
    """Монте-Карло оценка V^pi: усредняем возвраты G_t по всем посещениям s."""
    rng = np.random.default_rng(seed)
    total, count = defaultdict(float), defaultdict(int)
    starts = [s for s in STATES if s not in TERMINAL]   # из терминала не стартуем

    for _ in range(n_episodes):
        s0 = starts[rng.integers(len(starts))]
        G = 0.0
        # идём по траектории с конца: один проход даёт G_t сразу для всех шагов
        for s, _a, r in reversed(rollout(s0, policy, rng)):
            G = r + gamma * G
            total[s] += G
            count[s] += 1

    return {s: total[s] / count[s] if count[s] else 0.0 for s in STATES}


V_mc = estimate_V(policy_random, n_episodes=20000)
print({s: round(v, 3) for s, v in V_mc.items()})

### 5.2 Способ второй: аналитическая оценка (и тепловая карта)

Монте-Карло шумит и требует десятков тысяч эпизодов. Но `V^pi` можно посчитать
**точно**, без единого случайного числа. Уравнение Беллмана

```
V(s) = sum_a pi(a|s) * sum_s' P(s'|s,a) * [ R(s') + gamma * V(s') ]
```

связывает ценность клетки с ценностями соседей. Мы просто крутим это равенство
как правило пересчёта, пока значения не перестанут меняться. Для равномерной
политики `pi(a|s) = 1/4` это одна короткая функция.

In [ ]:
def evaluate_policy(gamma=0.95, tol=1e-9):
    """Аналитическая оценка V^pi для равномерной политики pi(a|s)=1/4:
    крутим уравнение Беллмана, пока значения не перестанут меняться."""
    V = {s: 0.0 for s in STATES}
    while True:
        new = {
            s: 0.0 if s in TERMINAL else sum(
                0.25 * p * (reward(s2) + gamma * V[s2])
                for a in ACTIONS
                for p, s2 in next_states(s, a)
            )
            for s in STATES
        }
        delta = max(abs(new[s] - V[s]) for s in STATES)
        V = new
        if delta < tol:
            return V


V_pi = evaluate_policy(gamma=0.95)
print({s: round(v, 3) for s, v in V_pi.items()})

Теперь — **та самая тепловая карта из урока**. Функция `plot_V` рисует ценность
всех девяти состояний: цвет клетки — её `V`, подпись — точное значение. Мы будем
переиспользовать её до конца ноутбука, поэтому у неё есть параметры `ax` и
`vmin`/`vmax` — они понадобятся, когда мы захотим положить несколько карт рядом
в одной цветовой шкале.

In [ ]:
def plot_V(V, title="V^pi, равномерная политика", ax=None,
           vmin=None, vmax=None, fontsize=11):
    """Тепловая карта ценности состояний на сетке 3x3."""
    grid = np.array([[V[(r, c)] for c in range(GRID)] for r in range(GRID)])
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))

    im = ax.imshow(grid, cmap="viridis", vmin=vmin, vmax=vmax)
    for (r, c), v in np.ndenumerate(grid):
        mark = {BATTERY: "батарея", PUDDLE: "лужа"}.get((r, c), "")
        ax.text(c, r, f"{mark}\n{v:+.2f}", ha="center", va="center",
                color="white", fontsize=fontsize)

    ax.set_title(title)
    ax.set_xticks(range(GRID)); ax.set_yticks(range(GRID))
    ax.set_xlabel("столбец"); ax.set_ylabel("строка")
    return im


im = plot_V(V_pi)
plt.colorbar(im, label="ценность состояния")
plt.show()

**Как это читать.** Терминальные клетки по соглашению имеют `V = 0` — из них уже
ничего не происходит. Все остальные значения отрицательные: случайный бот
блуждает долго, копит штрафы по `-0.02` и регулярно падает в лужу. Самые тёплые
клетки — соседние со станцией: оттуда есть шанс дойти до `+1` быстро.

Сверим два способа: аналитика и Монте-Карло должны сойтись.

In [ ]:
print(f"{'клетка':<10}{'Монте-Карло':>13}{'аналитика':>12}{'разница':>10}")
for s in STATES:
    print(f"{str(s):<10}{V_mc[s]:>13.3f}{V_pi[s]:>12.3f}{abs(V_mc[s] - V_pi[s]):>10.3f}")

gap = max(abs(V_mc[s] - V_pi[s]) for s in STATES)
print(f"\nмаксимальное расхождение: {gap:.3f} — это шум Монте-Карло, "
      f"он падает как 1/sqrt(число эпизодов)")

## § 6. Функция Q: сколько «стоит» действие в клетке

`V` отвечает на вопрос «насколько хороша клетка». Но чтобы **выбрать действие**,
нужен вопрос точнее: «насколько хорошо сделать вот этот шаг из вот этой клетки».
Это и есть `Q^pi(s, a)`.

Если `V^pi` уже оценён, то `Q^pi` получается буквально одним шагом «заглянуть
вперёд»: усредняем «мгновенная награда плюс gamma * V(следующего)» по переходам
из § 2.

In [ ]:
def Q_from_V(V, gamma=0.95):
    """Q(s, a) = сумма p * (r(s') + gamma * V(s')) по всем переходам из § 2."""
    return {
        (s, a): sum(p * (reward(s2) + gamma * V[s2]) for p, s2 in next_states(s, a))
        for s in STATES
        for a in ACTIONS
    }


Q_pi = Q_from_V(V_pi)
print("Q((2,1), ·):", {NAMES[a]: round(Q_pi[((2, 1), a)], 3) for a in ACTIONS})
print("Q((0,1), ·):", {NAMES[a]: round(Q_pi[((0, 1), a)], 3) for a in ACTIONS})

Обратите внимание на два случая выше: из клетки `(2, 1)` шаг **вправо** ведёт
прямо к станции и заметно выигрывает у остальных, а из клетки `(0, 1)` шаг
**вниз** ведёт в лужу и оказывается худшим. Именно эту разницу мы дальше и
превратим в политику.

Нарисуем все четыре карты — по одной на действие, в общей цветовой шкале.

In [ ]:
def plot_Q(Q, title="Q^pi(s, a) — по одной карте на действие"):
    """Четыре тепловые карты Q, по одной на действие, в общей шкале."""
    vals = np.array(list(Q.values()))
    vmin, vmax = vals.min(), vals.max()

    fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.6))
    for a, ax in zip(ACTIONS, axes):
        grid = np.array([[Q[((r, c), a)] for c in range(GRID)] for r in range(GRID)])
        im = ax.imshow(grid, cmap="viridis", vmin=vmin, vmax=vmax)
        for (r, c), v in np.ndenumerate(grid):
            ax.text(c, r, f"{v:+.2f}", ha="center", va="center",
                    color="white", fontsize=9)
        ax.set_title(NAMES[a])
        ax.set_xticks(range(GRID)); ax.set_yticks(range(GRID))

    fig.suptitle(title)
    fig.colorbar(im, ax=axes, shrink=0.85, label="Q(s, a)")
    plt.show()


plot_Q(Q_pi)

## § 7. Value Iteration: находим оптимальную политику

До сих пор мы **оценивали** заданную политику. Теперь будем её **улучшать**.

Идея в одну строку: вместо усреднения по действиям берём **максимум** по
действиям. Раз мы всё равно вольны выбирать `a`, естественно брать то, после
которого живём лучше всего:

```
V*(s)    = max_a Q*(s, a)
Q*(s, a) = sum_s' P(s'|s,a) * [ R(s') + gamma * V*(s') ]
```

Второе выражение — это ровно наш `Q_from_V` из § 6. Значит, никакого нового кода
писать не надо: вся итерация — это lookahead плюс `max`.

Заодно копим `history` — снимок `V` после каждой итерации. По нему мы и увидим
**процесс обучения**.

In [ ]:
def value_iteration(gamma=0.95, tol=1e-9):
    """V*(s) = max_a Q*(s, a). Считать Q из V мы уже умеем (§ 6),
    так что вся итерация — это lookahead плюс max."""
    V = {s: 0.0 for s in STATES}
    residuals, history = [], [V]

    while True:
        Q = Q_from_V(V, gamma)
        new = {s: 0.0 if s in TERMINAL else max(Q[(s, a)] for a in ACTIONS)
               for s in STATES}                       # V* на терминале = 0

        residuals.append(max(abs(new[s] - V[s]) for s in STATES))
        V = new
        history.append(V)                             # снимок для картинки обучения
        if residuals[-1] < tol:
            return V, residuals, history


def greedy_policy(V, gamma=0.95):
    """Оптимальная политика — просто argmax по Q, посчитанному из V*."""
    Q = Q_from_V(V, gamma)
    return {s: max(ACTIONS, key=lambda a: Q[(s, a)])
            for s in STATES if s not in TERMINAL}


V_star, residuals, history = value_iteration()
pi_star = greedy_policy(V_star)

print(len(residuals), "итераций до сходимости")
print({s: NAMES[a] for s, a in pi_star.items()})

### 7.1 Процесс обучения: как сходится Value Iteration

Первый график — невязка `max |V_{k+1} - V_k|` по итерациям в лог-шкале. Прямая
линия на логарифмическом графике означает **геометрическую** сходимость: каждая
итерация уменьшает ошибку в постоянное число раз. Это не совпадение — оператор
Беллмана является сжатием с коэффициентом не больше `gamma`, и именно поэтому
алгоритм всегда сходится, из какого бы начального `V` мы ни стартовали.

`gamma` здесь — только **гарантия сверху**. На RoomBot фактический множитель
заметно меньше: эпизоды заканчиваются в терминальных клетках, и вклад далёкого
будущего гаснет быстрее, чем просто `gamma` за шаг. Первые итерации вообще идут
рывками — фронт «света» от станции ещё только расходится по комнате, — и лишь
потом кривая выходит на ровный геометрический режим.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.semilogy(range(1, len(residuals) + 1), residuals, "o-", color="tab:cyan", ms=4)
ax.axhline(1e-9, ls="--", lw=1, color="tab:red", label="порог остановки tol")
ax.set_xlabel("итерация k")
ax.set_ylabel("max |V(k+1) - V(k)|")
ax.set_title("Сходимость Value Iteration (лог-шкала)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

ratio = residuals[-2] / residuals[-3]        # берём хвост: там режим уже установился
print(f"фактический множитель сходимости = {ratio:.3f} "
      f"(теория гарантирует не хуже gamma = 0.95)")

### 7.2 Процесс обучения: как «разгорается» карта ценности

А это то же самое, но глазами. На нулевой итерации все клетки нули — агент не
знает ничего. Потом «свет» от станции `+1` растекается по комнате шаг за шагом:
сначала загорается её сосед, потом сосед соседа, и так пока вся карта не
стабилизируется. Все кадры — в одной цветовой шкале, поэтому рост видно честно.

In [ ]:
frames = sorted({0, 1, 2, 3, 5, 8, 14, len(history) - 1})
frames = [f for f in frames if f < len(history)]

vmin, vmax = min(V_star.values()), max(V_star.values())

fig, axes = plt.subplots(1, len(frames), figsize=(2.8 * len(frames), 3.0))
for k, (f, ax) in enumerate(zip(frames, axes)):
    im = plot_V(history[f], title=f"итерация {f}", ax=ax,
                vmin=vmin, vmax=vmax, fontsize=8)
    ax.set_xlabel("")
    if k:                      # подписи оси Y — только у первого кадра
        ax.set_ylabel(""); ax.set_yticks([])

fig.suptitle("Обучение: как V растёт от нулей до V*", y=1.04)
fig.colorbar(im, ax=axes, shrink=0.85, label="V")
plt.show()

### 7.3 Результат: V* и оптимальная политика

Финальная карта с наложенными стрелками — это и есть ответ на задачу урока.
Стрелка в клетке показывает `argmax_a Q*(s, a)`: куда идти, чтобы жить лучше
всего. Сравните с политикой «вправо, потом вниз», которую мы в § 3 зашивали
руками, — Value Iteration нашла разумную стратегию сама, из одной лишь модели
среды.

In [ ]:
def plot_policy(V, pi, title="V* и оптимальная политика"):
    """Тепловая карта V со стрелками выбранных действий."""
    fig, ax = plt.subplots(figsize=(5.0, 4.6))
    im = plot_V(V, title=title, ax=ax)

    for (r, c), a in pi.items():
        dr, dc = MOVES[a]
        ax.arrow(c, r + 0.30, dc * 0.28, dr * 0.28,
                 head_width=0.11, head_length=0.10,
                 fc="white", ec="white", lw=1.4, zorder=5)

    fig.colorbar(im, ax=ax, shrink=0.82, label="V*")
    plt.show()


plot_policy(V_star, pi_star)

### 7.4 Случайный бот против оптимального — на одной шкале

Две карты рядом в общей цветовой шкале. Разница в магнитуде видна сразу: у
случайного бота вся комната «в минусе», у оптимального — вся в плюсе. Это и есть
то, что дало обучение.

In [ ]:
all_vals = list(V_pi.values()) + list(V_star.values())
vmin, vmax = min(all_vals), max(all_vals)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
plot_V(V_pi, title="V^pi — случайная политика", ax=axes[0], vmin=vmin, vmax=vmax)
im = plot_V(V_star, title="V* — оптимальная политика", ax=axes[1], vmin=vmin, vmax=vmax)
axes[1].set_ylabel(""); axes[1].set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.85, label="ценность состояния")
plt.show()

print(f"средняя ценность клетки: случайная {np.mean(list(V_pi.values())):+.3f}, "
      f"оптимальная {np.mean(list(V_star.values())):+.3f}")

## § 8. Что делает gamma

`gamma` — это «насколько агента волнует будущее». При `gamma` около нуля он видит
только следующий шаг, и почти вся комната для него одинаково серая. При `gamma`
около единицы сигнал от станции доходит до самых дальних углов.

Прогоним Value Iteration с несколькими `gamma` и положим карты рядом.

In [ ]:
gammas = [0.1, 0.5, 0.8, 0.95, 0.99]
results = {g: value_iteration(gamma=g) for g in gammas}

fig, axes = plt.subplots(1, len(gammas), figsize=(2.9 * len(gammas), 3.0))
for k, (g, ax) in enumerate(zip(gammas, axes)):
    plot_V(results[g][0], title=f"gamma = {g}", ax=ax, fontsize=8)   # своя шкала у каждой
    ax.set_xlabel("")
    if k:
        ax.set_ylabel(""); ax.set_yticks([])

fig.suptitle("V* при разных gamma", y=1.04)
plt.show()

for g in gammas:
    V_g, res_g, _ = results[g]
    print(f"gamma = {g:<5} -> V*(0,0) = {V_g[(0, 0)]:+.3f}, "
          f"итераций до сходимости: {len(res_g)}")

Видно два эффекта сразу. Во-первых, при маленькой `gamma` вся комната почти
серая: агенту важен только следующий шаг, и `V*(0,0)` едва отличается от нуля.
Во-вторых, чем больше `gamma`, тем больше итераций нужно: сигнал от станции
должен успеть добежать до дальних клеток, а вклад далёкого будущего затухает
медленнее. Дальнозоркость стоит вычислений — это первый из компромиссов, которые
будут преследовать нас весь курс.

## § 9. То же самое на PyTorch (опционально)

Переход на тензоры — это **не** другое решение, это та же математика,
переписанная так, чтобы одно матричное умножение делало работу всего тройного
цикла. Таблицу переходов один раз раскладываем в тензор `P[s, a, s']` — после
этого весь алгоритм умещается в три строки цикла.

Зачем это нужно, если RoomBot и так считается мгновенно:

* **батчинг** — в реальных задачах состояний миллионы, и векторизация ускоряет
  в сотни раз;
* **autograd бесплатно** — когда в уроке 1.6 `V` станет нейросетью, тот же граф
  операций посчитает градиенты без единой доп. строки;
* **GPU в одну строку** — `P = P.to("cuda")`.

Если `torch` не установлен, ячейка ниже аккуратно пропустится.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    print("torch не установлен — раздел пропущен, на остальной ноутбук это не влияет")

if torch is not None:
    # Та же модель из § 2, но разложенная по тензорам. Строится один раз.
    IDX = {s: i for i, s in enumerate(STATES)}
    P = torch.zeros(9, 4, 9)                                      # P[s, a, s']
    for s in STATES:
        for a in ACTIONS:
            for p, s2 in next_states(s, a):
                P[IDX[s], a, IDX[s2]] += p

    R = torch.tensor([reward(s) for s in STATES])                 # R[s']
    ALIVE = torch.tensor([0.0 if s in TERMINAL else 1.0 for s in STATES])

    def value_iteration_torch(gamma=0.95, tol=1e-9):
        V = torch.zeros(9)
        residuals = []

        while True:
            # Q[s, a] = sum_s' P[s, a, s'] * (R[s'] + gamma * V[s']) — один matmul
            # вместо всего тройного цикла из версии выше.
            V_new = (P @ (R + gamma * V)).max(dim=1).values * ALIVE

            residuals.append((V_new - V).abs().max().item())
            V = V_new
            if residuals[-1] < tol:
                return V, residuals

    V_torch, res_torch = value_iteration_torch()
    print(V_torch.reshape(3, 3).numpy().round(3))

    diff = max(abs(V_star[s] - V_torch[IDX[s]].item()) for s in STATES)
    print(f"\nмаксимальное расхождение со словарной версией: {diff:.2e}")
    assert diff < 1e-6, "реализации разошлись — где-то ошибка"
    print("две реализации дают один и тот же ответ")

## § 10. Проверка: а политика правда лучше?

Формулы формулами, но итоговая проверка любой RL-системы одна — запустить агента
в среде и посмотреть на средний возврат. Прогоним по 5000 эпизодов каждой
политикой из одинаковых стартов.

In [ ]:
def average_return(policy, n_episodes=5000, gamma=0.95, seed=1):
    """Средний G_0 по эпизодам со случайных нетерминальных стартов."""
    rng = np.random.default_rng(seed)
    starts = [s for s in STATES if s not in TERMINAL]
    total = 0.0
    for _ in range(n_episodes):
        s0 = starts[rng.integers(len(starts))]
        total += discounted_return(rollout(s0, policy, rng), gamma)
    return total / n_episodes


def make_policy(table):
    """Превращает словарь «клетка -> действие» в функцию policy(s, rng)."""
    return lambda s, rng: table[s]


scores = {
    "случайная (v1)":            average_return(policy_random),
    "рукописная вправо-вниз":    average_return(make_policy(fixed_policy)),
    "оптимальная (Value Iter.)": average_return(make_policy(pi_star)),
}

for name, g in scores.items():
    print(f"{name:<28} средний G_0 = {g:+.3f}")

fig, ax = plt.subplots(figsize=(6.8, 3.2))
names = list(scores)[::-1]
ax.barh(names, [scores[n] for n in names],
        color=["tab:cyan", "tab:purple", "tab:gray"])
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("средний возврат G_0 за эпизод")
ax.set_title("Проверка симуляцией: 5000 эпизодов на политику")
plt.show()

## Итоги

Что мы сделали за один ноутбук:

1. Формализовали комнату как MDP — пятёрку `(S, A, P, R, gamma)`. Вся модель
   среды уместилась в три функции: `move`, `next_states`, `reward`.
2. Увидели, что политика — это просто правило выбора действия, и она бывает
   случайной, детерминированной или стохастической.
3. Оценили `V^pi` двумя независимыми способами — Монте-Карло и уравнением
   Беллмана — и убедились, что они сходятся к одному.
4. Перешли от `V` к `Q` одним шагом lookahead и увидели, что `Q` — это уже
   готовая инструкция к действию.
5. Заменили усреднение по действиям на максимум и получили Value Iteration.
   Посмотрели и на процесс (кривая сходимости плюс раскадровка карты), и на
   результат (`V*` со стрелками политики).
6. Проверили ответ симуляцией — и повторили всё на тензорах PyTorch.

**Главный вывод.** Value Iteration требует знать модель среды: функции `P` и `R`
должны быть у нас на руках. В настоящих задачах их нет — Unity-агент не выдаёт
таблицу переходов своего мира. Именно поэтому в уроке 1.4 мы выбросим модель и
научимся строить ту же `Q` **из опыта**: это и будет Q-Learning.

### Домашнее задание

1. **Переставьте лужу** в `(0, 1)` и пересчитайте `V*`. Как изменилась политика
   в верхнем ряду?
2. **Сделайте пол скользким по-настоящему:** `SLIP = 0.5`. Останется ли
   оптимальная политика прежней? А `V*`?
3. **Уберите штраф за шаг** (`-0.02` в `0.0`) и поставьте `gamma = 1.0`.
   Value Iteration перестанет сходиться — объясните почему.
4. **Сравните с Policy Iteration:** напишите цикл «оценить текущую политику через
   `evaluate_policy`, улучшить через `greedy_policy`» и посмотрите, за сколько
   шагов он придёт к той же `pi_star`.
5. **Увеличьте комнату** до `GRID = 5`, раскидайте несколько луж и станций.
   Весь код выше должен заработать без единой правки — проверьте это.